# K-Means Clustering

**Course**: CMOR 438 / INDE 577 — Data Science & Machine Learning  
**Dataset**: International Football Results (1872–present)  
**Author**: Neriah29  

---

## The Shift to Unsupervised Learning

Every algorithm so far had **labels** — we knew which matches were home wins. The model's job was to learn to predict those labels.

Unsupervised learning removes the labels entirely:

> *Find structure in the data without being told what to look for.*

K-Means answers the question: **"Are there natural groups in this data?"**

---

## How K-Means Works

1. **Initialize** K centroids (using K-Means++ for smart placement)
2. **Assign** each point to its nearest centroid
3. **Update** each centroid to the mean of its assigned points
4. **Repeat** until centroids stop moving

### What it minimizes

$$\text{Inertia} = \sum_{i} ||x_i - \mu_{c_i}||^2$$

Total squared distance from each point to its assigned centroid. Lower = tighter, more compact clusters.

### K-Means++ Initialization

Pure random initialization often produces poor clusters. K-Means++ spreads the initial centroids out by picking each new centroid with probability proportional to its squared distance from existing centroids — much more likely to land in different natural cluster regions.

### Why run multiple times (n_init)?

K-Means can converge to local minima — different starting positions give different final clusters. Running n_init times with different initializations and keeping the best (lowest inertia) gives a more reliable result.

---

## Evaluation Without Labels

Since there are no labels, we can't compute accuracy. Instead:

- **Inertia**: lower is better — measures how tight the clusters are
- **Silhouette score**: ranges from -1 to +1 — measures how well separated clusters are from each other
- **Elbow method**: plot inertia vs K to find the natural number of clusters


---
## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.preprocessing import StandardScaler

import sys
sys.path.insert(0, '../../src')
from football_ml.unsupervised_learning.kmeans import KMeans

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
SEED = 42

---
## 1. Visualise K-Means on Toy Data

In [ ]:
rng = np.random.default_rng(0)
X0 = rng.normal(loc=[0, 0],   scale=0.4, size=(50, 2))
X1 = rng.normal(loc=[4, 0],   scale=0.4, size=(50, 2))
X2 = rng.normal(loc=[2, 3.5], scale=0.4, size=(50, 2))
X_toy = np.vstack([X0, X1, X2])

m_toy = KMeans(k=3, n_init=5, random_state=SEED).fit(X_toy)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Before clustering
axes[0].scatter(X_toy[:, 0], X_toy[:, 1], color='gray', s=20, alpha=0.6)
axes[0].set_title('Data — No Labels', fontsize=12, fontweight='bold')

# After clustering
colors = ['#4878CF', '#E87272', '#2ecc71']
for j in range(3):
    mask = m_toy.labels_ == j
    axes[1].scatter(X_toy[mask, 0], X_toy[mask, 1],
                    color=colors[j], s=20, alpha=0.7, label=f'Cluster {j}')
axes[1].scatter(m_toy.centroids_[:, 0], m_toy.centroids_[:, 1],
                color='black', marker='X', s=200, zorder=5, label='Centroids')
axes[1].set_title('After K-Means (K=3)', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('K-Means Clustering — Toy Example', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Inertia: {m_toy.inertia_:.2f}')
print(f'Converged in {m_toy.n_iter_} iterations')

---
## 2. Load & Engineer Features

In [ ]:
df = pd.read_csv('../../data/results.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df['goal_diff'] = df['home_score'] - df['away_score']
df['home_win']  = (df['home_score'] > df['away_score']).astype(int)
df['total_goals'] = df['home_score'] + df['away_score']

def compute_team_rolling_stats(df, window=10):
    home_log = df[['date', 'home_team', 'home_score', 'away_score']].copy()
    home_log.columns = ['date', 'team', 'scored', 'conceded']
    away_log = df[['date', 'away_team', 'away_score', 'home_score']].copy()
    away_log.columns = ['date', 'team', 'scored', 'conceded']
    team_log = pd.concat([home_log, away_log]).sort_values('date').reset_index(drop=True)
    team_log['rolling_scored'] = (
        team_log.groupby('team')['scored']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    team_log['rolling_conceded'] = (
        team_log.groupby('team')['conceded']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    return team_log.drop_duplicates(subset=['date', 'team'], keep='last').set_index(['date', 'team'])

team_stats = compute_team_rolling_stats(df)

def get_stat(row, team_col, stat_col):
    try:
        return team_stats.loc[(row['date'], row[team_col]), stat_col]
    except KeyError:
        return np.nan

df['home_goals_rolling']    = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_scored'), axis=1)
df['home_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_conceded'), axis=1)
df['away_goals_rolling']    = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_scored'), axis=1)
df['away_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_conceded'), axis=1)

home_wins = df.groupby('home_team').apply(lambda g: (g['home_score'] > g['away_score']).mean()).rename('home_win_rate')
away_wins = df.groupby('away_team').apply(lambda g: (g['away_score'] > g['home_score']).mean()).rename('away_win_rate')
df = df.join(home_wins, on='home_team').join(away_wins, on='away_team')
df['neutral'] = df['neutral'].astype(int)

feature_cols = [
    'home_goals_rolling', 'away_goals_rolling',
    'home_conceded_rolling', 'away_conceded_rolling',
    'home_win_rate', 'away_win_rate', 'neutral'
]
df_clean = df[feature_cols + ['goal_diff', 'home_win', 'total_goals']].dropna()
print(f'Dataset: {df_clean.shape[0]} matches')

---
## 3. Prepare Data — Scale First

In [ ]:
X = df_clean[feature_cols].values

# Scaling is critical — K-Means uses distance, same reason as KNN
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

# Use a sample for speed during elbow/silhouette computation
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(X_sc), size=5000, replace=False)
X_sample = X_sc[sample_idx]
print(f'Using {len(X_sample)} samples for clustering analysis')

---
## 4. Elbow Method — Finding the Right K

In [ ]:
k_values = list(range(2, 11))
inertias     = []
silhouettes  = []

for k in k_values:
    m = KMeans(k=k, n_init=3, random_state=SEED).fit(X_sample)
    inertias.append(m.inertia_)
    silhouettes.append(m.silhouette_score(X_sample))
    print(f'K={k:2d} | Inertia: {m.inertia_:10.1f} | Silhouette: {silhouettes[-1]:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(k_values, inertias, 'o-', color='#4878CF', linewidth=2)
axes[0].set_xlabel('K', fontsize=12)
axes[0].set_ylabel('Inertia', fontsize=12)
axes[0].set_title('Elbow Method', fontsize=12, fontweight='bold')

best_k_sil = k_values[np.argmax(silhouettes)]
axes[1].plot(k_values, silhouettes, 'o-', color='#E87272', linewidth=2)
axes[1].axvline(best_k_sil, color='black', linestyle='--', linewidth=1.2,
                label=f'Best K={best_k_sil}')
axes[1].set_xlabel('K', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score vs K', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

The **elbow** is where the inertia curve bends — adding more clusters beyond that point gives diminishing returns. The **silhouette score** peaks at the K that produces the best-separated clusters.

---
## 5. Fit Final Model

In [ ]:
best_k = best_k_sil
model = KMeans(k=best_k, n_init=10, random_state=SEED).fit(X_sample)

print(f'K = {best_k}')
print(f'Inertia:         {model.inertia_:.2f}')
print(f'Silhouette:      {model.silhouette_score(X_sample):.3f}')
print(f'Iterations:      {model.n_iter_}')
print(f'Cluster sizes:   {dict(zip(*np.unique(model.labels_, return_counts=True)))}')

---
## 6. Convergence Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(model.inertia_history_, color='#4878CF', linewidth=2)
ax.fill_between(range(len(model.inertia_history_)),
                model.inertia_history_, alpha=0.1, color='#4878CF')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Inertia', fontsize=12)
ax.set_title('K-Means Convergence', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Cluster Profiles — What Did K-Means Find?

In [ ]:
df_sample = df_clean.iloc[sample_idx].copy()
df_sample['cluster'] = model.labels_

# Profile each cluster
profile = df_sample.groupby('cluster')[feature_cols + ['goal_diff', 'home_win', 'total_goals']].mean()
print('Cluster profiles (mean values):')
print(profile.round(3).to_string())

In [ ]:
# Heatmap of cluster centroids (in feature space)
centroid_df = pd.DataFrame(
    model.centroids_,
    columns=feature_cols,
    index=[f'Cluster {j}' for j in range(best_k)]
)

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(centroid_df, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, annot_kws={'size': 8})
ax.set_title('Cluster Centroids — Feature Values (scaled)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

This heatmap shows where each cluster sits in feature space. Positive values (red) mean that cluster has above-average values for that feature; negative (blue) means below-average. You can start to interpret what each cluster represents — e.g. a cluster with high home_goals_rolling and high home_win_rate might represent "strong home team" matches.

---
## 8. Discussion — What Does Clustering Tell Us?

### What K-Means found

Without any labels, K-Means discovered groups of matches that naturally share similar feature profiles. These clusters might represent different types of match contexts — evenly matched games, dominant home team games, strong away team games, etc.

### Limitations

1. **K must be chosen upfront** — the elbow and silhouette methods help but don't give a definitive answer
2. **Assumes spherical clusters** — K-Means draws circular (Euclidean) boundaries. Elongated or irregular clusters are poorly handled
3. **Sensitive to initialization** — K-Means++ and multiple restarts mitigate this but don't eliminate it
4. **Scaling required** — like KNN, distance-based so features must be on the same scale
5. **Cluster labels are arbitrary** — cluster 0 today might be cluster 2 tomorrow with a different seed

### What comes next

DBSCAN (next algorithm) fixes two of these: it doesn't require specifying K upfront, and it can find arbitrarily shaped clusters. It also explicitly handles noise/outliers — something K-Means cannot do.

---
## Summary

| | |
|---|---|
| **Algorithm type** | Unsupervised clustering |
| **Approach** | Iterative centroid assignment and update |
| **Minimizes** | Inertia — total within-cluster squared distance |
| **Key parameter** | K — number of clusters (set by you) |
| **Initialization** | K-Means++ for smart centroid placement |
| **Evaluation** | Elbow method, silhouette score |
| **Needs scaling** | Yes — distance-based |
| **Key limitation** | Assumes spherical clusters, K must be specified |
